In [2]:
import math
import random
import csv
import pandas as pd
import numpy as np
import itertools
from tqdm.notebook import tqdm

# Measuring kin term granularity

The following code extracts kinship term data from Kinbank and calculates the kin term granularity of each language: the number of unique kin terms it uses to denote a semantic space of 48 kin, as a proportion. It generates csv dataframes including the language, language family, and lexicon size.

The second part of this notebook contains code for building simulated kinship systems by sampling from the possible kinship categories that exist in the Kinbank data. These systems are plausible alternatives to natural kinship terminology systems, but may not emerge in natural language. Using these systems, we can show whether or not natural language have coarser- or finer-grain kin term granularity than we would expect if there were under no constraints.

We want to work out kin term granularity over a particular kinship space (so over a subset of relatives), rather than just count the number of unique entries in the lexicon. So first, we need to define that kinship space.

Our semantic space is given below in shorthand: each entry in the following list corresponds to a kin type, e.g. mMeB = man's mother's elder brother. Kin terms are stored under these shorthands in Kinbank, making it easier for us to pick out the corresponding data later.

In [3]:
full_kinship_space = [
    # G+2
    'mMM','mMF','mFM','mFF',
    
    # G+1
    'mM','mF',
    'mMeB','mMyB', 
    'mMeZ','mMyZ',
    'mFeB','mFyB',
    'mFeZ','mFyZ',
    
    # G0

    'meZ','myZ',
    'meB','myB',
    'mMeBD','mMyBD',
    'mMeBS','mMyBS',
    'mMeZD','mMyZD',
    'mMeZS','mMyZS',
    'mFeBD','mFyBD',
    'mFeBS','mFyBS',
    'mFeZD','mFyZD',
    'mFeZS','mFyZS',
    
    # G-1
    'mD','mS',
    'meZD','myZD',
    'meZS','myZS',
    'meBD','myBD',
    'meBS','myBS',
    
    # G-2
    'mDD','mDS',
    'mSD','mSS'
]


## Getting the data

First, we read in the Kinbank data.

In [4]:
kinbank = pd.read_csv('kinbank.csv')

The following function extracts the relevant data from the `kinbank` dataframe.

For a particular language (specified by its `glottocode`), and for each kin type listed in `kt.relatives` (imported from a separate file), create a key:value pair where the key is the kin type and the value is the kin term.

Return a dictionary of all key:value pairs.

In [5]:
def get_kin_terms(glottocode):
    language_data = kinbank[kinbank['Glottocode'] == glottocode]

    kinterms = {}

    for relative in full_kinship_space:
        row = language_data[language_data['Parameter_ID'] == relative]
        if row.empty:
            pass
        else:
            term = list(row['Form'])[-1]
            kinterms[relative] = term

    return kinterms

Create a list of all glottocodes so we can iterate over all languages later.

In [6]:
all_glottocodes = list(set(kinbank['Glottocode']))

## Extracting kin term granularity data

Then we write a function that extracts the terms for these kin types out of the kinbank data, counts them, and calculates the proportion of terms over types.

The reason we calculate granularity as a proportion (rather than a raw count) is because not all languages have complete data for all 48 kin types in the `full_kinship_space`. To maximise the number of languages we can include in our analysis, we calculate a proportion instead of a raw count.

`calculate_granularity` takes two arguments: a kinship space (in our case, `full_kinship_space`, but this could be any list of kin types), and a kinship system, a dictionary of kintype:kinterm mappings extracted from the Kinbank data.

In [7]:
def calculate_granularity(kinship_space,kinship_system):
    
    kinterms = []
    
    # for each kin type in the semantic space
    for kintype in kinship_space:
        
        # if there is an entry for that kin type in this kinship system
        if kintype in kinship_system:
                kinterms.append(kinship_system[kintype]) # then store that kin term in the kinterms list
                                
    if len(kinterms) > 0: # if there was at least one kin type in the kinship system
        return len(set(kinterms)) / len(kinterms) # return the proportion of terms over types      

Now we can calculate granularity across the whole dataset, and save the output to a CSV file. `granularity_across_languages` takes two arguments: a kinship space, and the filename for the output file.

In [12]:
def granularity_across_languages(kinship_space,filename):
    df = []

    for code in tqdm(all_glottocodes):

        language = list(kinbank[kinbank['Glottocode'] == code]['Name'])[0] # get the language name
        family = list(kinbank[kinbank['Glottocode'] == code]['Family'])[0] # get the family 

#         if code not in codes:
#             codes.append(code)

        ks = get_kin_terms(code)
            
        data = {}
        granularity = calculate_granularity(kinship_space,ks)
            
            
            
        data['language'] = language
        data['glottocode'] = code
        data['family'] = family
        data['prop'] = granularity
        data['n_kintypes'] = len(ks)

        df.append(data)
                
    pd.DataFrame(df).to_csv('../analysis/data/raw/' + filename + '.csv',index=False)

                
    return pd.DataFrame(df)
    

In [13]:
granularity_across_languages(full_kinship_space,'granularity')

  0%|          | 0/1127 [00:00<?, ?it/s]

,language,code,family,prop,n_kinterms
0,Kagulu,kagu1239,Atlantic-Congo,0.333333,42
1,Vaeakau-Taumako,pile1238,Austronesian,0.250000,48
2,Dogrib,dogr1252,NaN,0.500000,32
3,Qawasqar,qawa1238,NaN,0.375000,40
4,Mbyá Guaraní,mbya1239,Tupian,0.229167,48
...,...,...,...,...,...
1122,Abui,abui1241,NaN,0.406250,32
1123,Galibi Carib,gali1262,Cariban,0.390244,41
1124,Turung,turu1249,Tai-Kadai,0.687500,32
1125,Ngarinman,ngar1235,Pama-Nyungan,0.400000,40


# Simulating a kinship system

We have successfully calculated kin term granularity for natural languages, but how does the granularity of natural kinship terminologies compare to what is theoretically possible?

The following code generates simulated systems of kinship terminology by sampling from possible categories of kin from the natural language data.

Then, we calculate the granularity of those systems, and save that to a separate data file.

In order to do this, we need to take three key steps:
* For every term that exists in any language in our dataset, make a list of which relatives are referred to with that term.
* Iterate over the kin types in the `full kinship space`, assigning terms according to who is a possible co-referent.
* Sample until the `full kinship space` is filled.

First, let's write a function that takes one argument `ks`, a kinship system, and returns a list of all kin categories in that kinship system - i.e., a list of groups of indiviudals who share a term.

In [16]:
def get_categories(ks):

    categories = []
    terms = list(ks.values())

    for term in set(terms):
        category = []
        for key in ks:
            if term == ks[key]:
                category.append(key)
        categories.append(category)

    return categories
                

However, if we just sample full categories, we will very likely end up in a situation where we cannot completely fill the kinship space.

Instead, we want to assign terms to two individuals at a time in a pairwise fashion.

Using the function above, we work out all the *pairwise categories* that exist in natural kinship systems. That is, which pairs of individuals can share a term in any language?

For example, in English, the term 'uncle' denotes several pairwise categories: 
* (mother's older brother, mother's younger brother), 
* (mother's older brother, father's older brother), 
* (mother's older brother, father's younger brother), 
* (mother's younger brother, father's older brother), 
* (mother's younger brother, father's younger brother)

And in addition, any individual referent of 'uncle' can be in a pairwise category with themselves (which allows any individual to only share a term with themselves, e.g. 'mother':
* (mother's older brother, mother's older brother)
* (mother's younger brother, mother's younger brother),
* (father's older brother, father's older brother),
* (father's younger brother, father's younger brother).

`get_pairwise_cats` takes a single argument `ks`, a kinship system, and returns a list of all the pairwise categories in that language.

In [17]:
def get_pairwise_cats(ks):
    categories = []
#     all_pairs = list(itertools.combinations(full_kinship_space,2)) # get all possible pairs of kin types
        
    for r1 in full_kinship_space:
        for r2 in full_kinship_space:
            pair = [r1,r2]
            if pair[0] in ks and pair[1] in ks:
                if ks[pair[0]] == ks[pair[1]]:
                    categories.append((pair[0],pair[1]))
                else:
                    pass
                
    return categories

The delightfully named `cats_for_simulation()` makes a list of all pairwise categories that exist in all languages, including repeats. Later, our simulation will refer to this list when sampling. By including repeats the model is more likely to choose kin types that appear more often in the cross-linguistic data.

In [18]:
def cats_for_simulation():
    all_lang_cats = []
    
    for code in tqdm(all_glottocodes):
        ks = get_kin_terms(code)
        if len(ks.values()) < 48:
            pass
        else:
            all_lang_cats.append(get_pairwise_cats(ks))
        
    all_lang_cats = [i for j in all_lang_cats for i in j]
    
    return all_lang_cats  

In [19]:
all_cats = cats_for_simulation()

  0%|          | 0/1127 [00:00<?, ?it/s]

In [34]:
all_cats

[('mMM', 'mMM'),
 ('mMM', 'mFM'),
 ('mMF', 'mMF'),
 ('mMF', 'mFF'),
 ('mFM', 'mMM'),
 ('mFM', 'mFM'),
 ('mFF', 'mMF'),
 ('mFF', 'mFF'),
 ('mM', 'mM'),
 ('mF', 'mF'),
 ('mMeB', 'mMeB'),
 ('mMeB', 'mMyB'),
 ('mMyB', 'mMeB'),
 ('mMyB', 'mMyB'),
 ('mMeZ', 'mMeZ'),
 ('mMeZ', 'mMyZ'),
 ('mMyZ', 'mMeZ'),
 ('mMyZ', 'mMyZ'),
 ('mFeB', 'mFeB'),
 ('mFeB', 'mFyB'),
 ('mFyB', 'mFeB'),
 ('mFyB', 'mFyB'),
 ('mFeZ', 'mFeZ'),
 ('mFeZ', 'mFyZ'),
 ('mFyZ', 'mFeZ'),
 ('mFyZ', 'mFyZ'),
 ('meZ', 'meZ'),
 ('myZ', 'myZ'),
 ('meB', 'meB'),
 ('myB', 'myB'),
 ('mMeBD', 'mMeBD'),
 ('mMeBD', 'mMyBD'),
 ('mMeBD', 'mMeZD'),
 ('mMeBD', 'mMyZD'),
 ('mMeBD', 'mFeBD'),
 ('mMeBD', 'mFyBD'),
 ('mMeBD', 'mFeZD'),
 ('mMeBD', 'mFyZD'),
 ('mMyBD', 'mMeBD'),
 ('mMyBD', 'mMyBD'),
 ('mMyBD', 'mMeZD'),
 ('mMyBD', 'mMyZD'),
 ('mMyBD', 'mFeBD'),
 ('mMyBD', 'mFyBD'),
 ('mMyBD', 'mFeZD'),
 ('mMyBD', 'mFyZD'),
 ('mMeBS', 'mMeBS'),
 ('mMeBS', 'mMyBS'),
 ('mMeBS', 'mMeZS'),
 ('mMeBS', 'mMyZS'),
 ('mMeBS', 'mFeBS'),
 ('mMeBS', 'mFyBS'),


`simulate_kinship_system()` does just that: it takes the pairwise categories we have generated, and samples from this data until it has filled the entire kinship space. At this point, the model has generated a system that is entirely plausible given the kinds of kin that can be categorised in the world's languages, but may not actually be instantiated in any language. 

The function takes two arguments: `kinship_space` and `sim_type` - the latter is an optional parameter so that we can test different kinds of constraints on the model. As standard, the model will fill the kinship space in a random order - it shuffles the list of kin types that need to be assigned terms, and starts from the top of this random list. If `sim_type` is set to `close_first`, the model will assign terms to M, F, S, D, Z and B first.

In [20]:
def simulate_kinship_system(kinship_space,sim_type='random'):
    
    ks = {} # empty kinship system, to be populated as the model runs

    pairwise_categories = [] # list of pairwise categories selected by the model
        
    if sim_type == 'random':
        random.shuffle(kinship_space)
        
    elif sim_type == 'close_first':
        close_kin = ['mM','mF','mS','mD','meB','meZ','myB','myZ']
        kinship_space = [i for i in kinship_space if i not in close_kin]
        random.shuffle(kinship_space)
        kinship_space = close_kin + kinship_space
        
            
    for relative in kinship_space:
        possible_pairs = [pair for pair in all_cats if pair[0] == relative] # get the possible pairs of coreferents for that relative
        chosen_pair = random.choice(possible_pairs) # choose one at random
        pairwise_categories.append(chosen_pair) # append it to pairwise_categories
   
    # now we build the kinship system from the randomly selected pairs in pairwise_categories
    
    counter = 1 # the counter serves as the kin term for each iteration
    # first kin type is labelled 1, second is labelled 2, etc.

    
    for pair in pairwise_categories: # for each pair
        
        # if both pairs have already been assigned a term, move on
        if pair[0] in ks and pair[1] in ks:
            pass
        
        # if one member of the pair has been assigned a term, then assign the same term to the other member
        elif pair[0] in ks and pair[1] not in ks:
            ks[pair[1]] = ks[pair[0]]
            
        elif pair[1] in ks and pair[0] not in ks:
            ks[pair[0]] = ks[pair[1]]
        
        # if neither member has been assigned a term yet, assign a term to both
        else:
            ks[pair[0]] = counter
            ks[pair[1]] = counter
            
        counter += 1 # increase the counter for the next iteration
            
    return ks

In [49]:
simulate_kinship_system(full_kinship_space,'random')

{'mFyZD': 1,
 'mFeBD': 1,
 'myZS': 2,
 'myBS': 2,
 'meBD': 2,
 'meZ': 4,
 'mFeZD': 4,
 'mMeZ': 5,
 'mMyZ': 5,
 'mFyZ': 5,
 'meZS': 7,
 'meBS': 7,
 'mMeBS': 8,
 'mFeBS': 8,
 'mFyBD': 4,
 'mFM': 10,
 'mSD': 10,
 'mMyBS': 11,
 'mFyBS': 11,
 'mMyB': 13,
 'mFyB': 13,
 'mMeZS': 14,
 'mMeBD': 14,
 'mFF': 15,
 'mMF': 15,
 'mFeB': 13,
 'mSS': 17,
 'meB': 14,
 'mMyBD': 4,
 'mFeZS': 22,
 'mMeZD': 4,
 'mMeB': 13,
 'mDS': 26,
 'mMyZS': 11,
 'mS': 29,
 'mFeZ': 5,
 'mFyZS': 1,
 'myBD': 5,
 'myZ': 37,
 'meZD': 2,
 'mMyZD': 14,
 'mD': 42,
 'mMM': 15,
 'mF': 44,
 'myB': 45,
 'mDD': 46,
 'myZD': 47,
 'mM': 48}

Finally, we put all the bits together in one big function, create a bunch of kinship systems, measure their granularity, and save all the data at the end.

`full_simulation` takes four arguments: `times` is the number of kinship systems to be simulated, `kinship_space` is the kinship space, `filename` is the output file name, and `sim_type` is `random` (by default) or `close_first`.

In [21]:
def full_simulation(times,kinship_space,filename,sim_type='random'):
    
    df = [] # create an empty dataframe
    
    for i in tqdm(range(times)):
        
        sim_ks = simulate_kinship_system(kinship_space,sim_type)
        
        data = {}
        prop = calculate_granularity(kinship_space,sim_ks)
        
        data['language'] = 'simulation_' + str(i)
        data['glottocode'] = 'na'
        data['family'] = 'na'
        data['prop'] = prop
        data['n_kintypes'] = 48

        df.append(data)
                
    pd.DataFrame(df).to_csv('../analysis/data/raw/' + filename + '.csv',index=False)

    return pd.DataFrame(df)
        
    

In [120]:
full_simulation(1000,full_kinship_space,'simulation')

  0%|          | 0/1000 [00:00<?, ?it/s]

,language,code,family,prop
0,simulation_0,na,na,0.395833
1,simulation_1,na,na,0.416667
2,simulation_2,na,na,0.416667
3,simulation_3,na,na,0.479167
4,simulation_4,na,na,0.395833
...,...,...,...,...
995,simulation_995,na,na,0.395833
996,simulation_996,na,na,0.375000
997,simulation_997,na,na,0.416667
998,simulation_998,na,na,0.479167


In [22]:
# ordered simulation
full_simulation(1000,full_kinship_space,'ordered_simulation',sim_type='close_first')

  0%|          | 0/1000 [00:00<?, ?it/s]

,language,glottocode,family,prop,n_kintypes
0,simulation_0,na,na,0.395833,48
1,simulation_1,na,na,0.458333,48
2,simulation_2,na,na,0.458333,48
3,simulation_3,na,na,0.437500,48
4,simulation_4,na,na,0.416667,48
...,...,...,...,...,...
995,simulation_995,na,na,0.520833,48
996,simulation_996,na,na,0.458333,48
997,simulation_997,na,na,0.458333,48
998,simulation_998,na,na,0.458333,48


## Appendix B: Alternative model constraints

Our model makes very few assumptions about how kinship 'works' - it uses the cross-linguistic data to inform the kinds of distinctions that can be possible in a system of kinship terminology. However, because it only samples from possible categories, it is fairly constrained relative to alternative models.

Here, we generate data using two other models: one that assigns kin terms to kin types completely randomly, and one that assigns kin terms to kin types randomly, but weighted by the shared semantic content between kin types.

## Weighted model

The weighted model is similar, but requires a few more functions to set up. We need functions that determine the semantic similarity between pairs of kin types.

In this model, we incorporate four kinds of semantic similarity: similarity of generation, gender, side of the family, and relative age.

The following four functions take one argument, `relative`, and output the value of a feature for that kin type.

In [69]:
def determine_gender(relative):
    women = ['M','Z','D']
    men = ['F','B','S']

    final_letter = relative[-1]
    if final_letter in women:
        gender = 'f'
    elif final_letter in men:
        gender = 'm'
        
    return gender

In [70]:
def determine_side(relative):
    
    if relative[1] == 'S' or relative[1] == 'D':
        return 'E' # e for direct lineage from ego

    else:
        return relative[1]

In [71]:
def determine_generation(relative):
    
    if relative in ['mMM','mMF','mFM','mFF']:
        return 2
    
    if relative in ['mM','mF','mMeB','mMyB','mMeZ','mMyZ','mFeB','mFyB','mFeZ','mFyZ']:
        return 1
    
    if relative in ['meZ','myZ','meB','myB', 'mMeBD','mMyBD','mMeBS','mMyBS','mMeZD','mMyZD',
                    'mMeZS','mMyZS','mFeBD','mFyBD','mFeBS','mFyBS','mFeZD','mFyZD','mFeZS','mFyZS']:
        return 0
    if relative in ['mD','mS','meZD','myZD','meZS','myZS','meBD','myBD','meBS','myBS']:
        return -1
    
    if relative in ['mDD','mDS','mSD','mSS']:
        return -2

In [72]:
def determine_age(relative):
    
    if 'e' in relative:
        age = 'e'
    elif 'y' in relative:
        age = 'y'
    else:
        age = 'na'
    
    return age

Next we want to write a function that takes a pair of kin types `x` and `y`, determines the feature values for each, and calculates their semantic similarity based on how many features they share.

In [116]:
def feature_similarity(x,y):
    
    sem_distance = [0,0,0,0] # dummy array for shared features
        
    if determine_gender(x) == determine_gender(y):
            sem_distance[0] = 1
        
    if determine_side(x) == determine_side(y):
        sem_distance[1] = 1
        
    if determine_generation(x) == determine_generation(y):
        sem_distance[2] = 1
        
    if determine_age(x) != 'na' and determine_age(y) != 'na':
        if determine_age(x) == determine_age(y):
            sem_distance[3] = 1
                    
    return sum(sem_distance)

Now we need a way to convert semantic distance values into weights. `get_weights` takes two arguments: a kin type, and the list of all other kin types. The function determines how much semantic content is shared between `relative` and each other relative, and assigns a weight to each pair on the basis of their semantic distance.

In [114]:
def get_weights(relative,list_of_relatives):
    weights = []
    
    for relative2 in list_of_relatives:
        feature_sim = feature_similarity(relative,relative2)
        if feature_sim == 0:
            weights.append(1/48)
        if feature_sim == 1:
            weights.append(1/24)
        if feature_sim == 2:
            weights.append(1/12)
        if feature_sim == 3:
            weights.append(1/6),
        if feature_sim == 4:
            weights.append(1/3)
    
    return weights

## Random model

Like our reported model, the random model and the weighted model iterate over kin types and assigns terms in a pairwise fashion. The key difference from our main model is that rather than sampling plausible co-referents from the cross-linguistic data, the random model randomly selects a co-referent from among any and all other kin types.

`simulate_random_ks` takes a list of relatives (the full kinship space in our case) and parameter `weighted` that tells us whether we are doing a weighted simulation or not. The default here is False.

In [96]:
def simulate_random_ks(kinship_space, weighted=False):
    
    counter = 0 # index for assigning kin terms to types
    
    ks = {} # dummy kinship system to be populated
    
    random.shuffle(kinship_space)

    for relative in kinship_space: # for each relative
        
        if weighted:
            weights = get_weights(relative,kinship_space)
            
        else:
            weights = None
            
        coreferent = random.choices(kinship_space,weights)[0] # choose a relative as coreferent

       # if both pairs have already been assigned a term, move on
        if relative in ks and coreferent in ks:
            pass
        
        # if one member of the pair has been assigned a term, then assign the same term to the other member
        elif relative in ks and coreferent not in ks:
            ks[coreferent] = ks[relative]
            
        elif coreferent in ks and relative not in ks:
            ks[relative] = ks[coreferent]
        
        # if neither member has been assigned a term yet, assign a term to both
        else:
            ks[relative] = counter
            ks[coreferent] = counter

        counter += 1
    
    return ks

Put all the pieces together and save the granularity data to CSV:

In [97]:
def alternate_simulation(times,kinship_space,filename,weighted=False):
    
    df = [] # create an empty dataframe
    
    for i in tqdm(range(times)):
        
        sim_ks = simulate_random_ks(kinship_space,weighted)
        
        data = {}
        prop = calculate_granularity(kinship_space,sim_ks)
        
        data['language'] = 'simulation_' + str(i)
        data['glottocode'] = 'na'
        data['family'] = 'na'
        data['prop'] = prop
        data['n_kintypes'] = 48

        df.append(data)
                
    pd.DataFrame(df).to_csv('../analysis/data/raw/' + filename + '.csv',index=False)

    return pd.DataFrame(df)

In [93]:
# random model
alternate_simulation(1000,full_kinship_space,'stochastic_model')

  0%|          | 0/1000 [00:00<?, ?it/s]

,language,code,family,prop
0,simulation_0,na,na,0.312500
1,simulation_1,na,na,0.229167
2,simulation_2,na,na,0.270833
3,simulation_3,na,na,0.291667
4,simulation_4,na,na,0.333333
...,...,...,...,...
995,simulation_995,na,na,0.354167
996,simulation_996,na,na,0.312500
997,simulation_997,na,na,0.291667
998,simulation_998,na,na,0.270833


In [118]:
# weighted model
alternate_simulation(1000,full_kinship_space,'weighted_model',weighted=True)

  0%|          | 0/1000 [00:00<?, ?it/s]

,language,code,family,prop
0,simulation_0,na,na,0.354167
1,simulation_1,na,na,0.270833
2,simulation_2,na,na,0.354167
3,simulation_3,na,na,0.354167
4,simulation_4,na,na,0.312500
...,...,...,...,...
995,simulation_995,na,na,0.333333
996,simulation_996,na,na,0.333333
997,simulation_997,na,na,0.333333
998,simulation_998,na,na,0.354167
